# 🎯 실습 — 종합 파이프라인 (학생용)

전처리 → 벡터화 → 유사도 전체 파이프라인을 직접 구현합니다.

> **🖥️ 환경 설정**
> - 로컬 `UD_26` 실행: `uv run jupyter lab`
> - 로컬 `UD_26` 패키지 에러: `uv add pecab scikit-learn pandas`
> - Colab 패키지 에러: `!pip install pecab scikit-learn pandas -q`

> **⌨️ 단축키 안내**
> | 단축키 | 동작 |
> |--------|------|
> | `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
> | `Esc → A` | 위에 셀 삽입 |
> | `Esc → B` | 아래에 셀 삽입 |
> | `Esc → DD` | 셀 삭제 |

In [ ]:
# 패키지 설치가 필요할 때만 참고하세요.
# - 로컬 UD_26: uv add pecab scikit-learn pandas
# - Colab: !pip install pecab scikit-learn pandas -q

In [1]:
import re
import pandas as pd
from pecab import PeCab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pecab = PeCab()

## 📌 §340 종합 실습 — 진짜 중요한 단어를 찾아보자

이번 주에 배운 전체 파이프라인(전처리 → 벡터화 → 유사도)을 한 번에 실행합니다.

[→ §340 강의노트](../notes/UD-05-340__comprehensive-practice.md)

In [2]:
# 📌 §340 데이터 준비
corpus_raw = [
    "코로나 거리두기와 코로나 상생지원금 문의입니다.",
    "지하철 운행시간과 지하철 요금 문의입니다.",
    "지하철 승강장 문의입니다.",
    "택시 승강장 문의입니다.",
    "코로나 선별진료소 위치 문의입니다.",
    "버스 운행 시간 문의입니다."
]
print(f"문서 수: {len(corpus_raw)}")
# 예상 출력: 문서 수: 6

문서 수: 6


In [4]:
# 📌 §340 Pecab 전처리
corpus_clean = []
stop_tokens = {"와", "과", "입니다"}

for text in corpus_raw:
  text = re.sub(r"[^가-힣\s]", " ", text)
  tokens = [token for token in pecab.morphs(text) if len(token) > 1 and token not in stop_tokens]
  corpus_clean.append(" ".join(tokens))

for i, doc in enumerate(corpus_clean):
  print(f"문서 {i}: {doc}")

# 예상 출력:
#   문서 0: 코로나 거리두기 코로나 상생지원금 문의
#   문서 1: 지하철 운행시간 지하철 요금 문의
#   ...

문서 0: 거리 코로나 상생 지원금 문의
문서 1: 지하철 운행 시간 지하철 요금 문의
문서 2: 지하철 승강장 문의
문서 3: 택시 승강장 문의
문서 4: 코로나 선별 진료소 위치 문의
문서 5: 버스 운행 시간 문의


In [5]:
# 📌 §340 TF-IDF 벡터화
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_clean)

vocab = sorted(vectorizer.vocabulary_.keys())
print(f"단어 사전 ({len(vocab)}개): {vocab}")

# 예상 출력: 단어 사전 (N개): ['거리두기', '문의', '버스', ...]

단어 사전 (15개): ['거리', '문의', '버스', '상생', '선별', '승강장', '시간', '요금', '운행', '위치', '지원금', '지하철', '진료소', '코로나', '택시']


In [6]:
# 📌 §340 코사인 유사도
similarity_matrix = cosine_similarity(tfidf_matrix)

labels = [f"문서 {i}" for i in range(len(corpus_raw))]
similarity_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
print(similarity_df.round(3))

# 예상 출력: 6x6 유사도 행렬 (코로나 문서끼리 높은 유사도)

       문서 0   문서 1   문서 2   문서 3   문서 4   문서 5
문서 0  1.000  0.044  0.081  0.073  0.225  0.063
문서 1  0.044  1.000  0.543  0.063  0.044  0.423
문서 2  0.081  0.543  1.000  0.512  0.081  0.100
문서 3  0.073  0.063  0.512  1.000  0.073  0.090
문서 4  0.225  0.044  0.081  0.073  1.000  0.063
문서 5  0.063  0.423  0.100  0.090  0.063  1.000
